# Phase 1 · Step 2 — PDF ingestion + chunking

Loads a PDF page-by-page with LlamaIndex's `PDFReader`, then splits those
pages into overlapping chunks with `SentenceSplitter`. See
`learning/documents/01_pdf_ingestion_chunking.md` for the why behind these
choices.

No real paper is committed to the repo — `generate_sample_pdf.py` builds a
small throwaway one locally so this notebook is reproducible from code.

In [1]:
from pathlib import Path

from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader

pdf_path = generate()
pdf_path

PosixPath('/Users/rtlecl910010/Desktop/Projects/Project/papermind/backend/notebooks/data/sample.pdf')

## Load

`PDFReader` turns the PDF into one `Document` per page, keeping the page
number in `metadata["page_label"]`.

In [2]:
documents = PDFReader().load_data(file=pdf_path)
print(f"Loaded {len(documents)} page-documents")
for doc in documents:
    print(doc.metadata, repr(doc.text[:70]))

Loaded 4 page-documents
{'page_label': '1', 'file_name': 'sample.pdf'} 'Abstract\nThis paper studies retrieval-augmented generation for questio'
{'page_label': '2', 'file_name': 'sample.pdf'} '1. Introduction\nLarge language models can answer questions fluently bu'
{'page_label': '3', 'file_name': 'sample.pdf'} '2. Method\nDocuments are split into overlapping chunks of roughly five '
{'page_label': '4', 'file_name': 'sample.pdf'} '3. Results\nAcross a held-out set of research papers, grounded answers '


## Chunk

`SentenceSplitter` breaks on sentence boundaries, keeps `page_label` on
every chunk it produces, and only hard-cuts if a single sentence is longer
than `chunk_size`.

In [3]:
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")
for node in nodes:
    print(f"--- page {node.metadata['page_label']} ---")
    print(node.get_content())
    print()

Produced 4 chunks from 4 pages
--- page 1 ---
Abstract
This paper studies retrieval-augmented generation for question answering over research
papers. We show that combining dense retrieval with a re-ranking stage improves citation
accuracy compared to retrieval alone.

--- page 2 ---
1. Introduction
Large language models can answer questions fluently but often fabricate details not
present in the source document. Grounding each answer in retrieved passages, and citing
the page each passage came from, lets a reader verify the answer directly.

--- page 3 ---
2. Method
Documents are split into overlapping chunks of roughly five hundred tokens. Each chunk is
embedded and stored alongside the page number it was extracted from, so retrieval results
can always be traced back to a specific page in the original PDF.

--- page 4 ---
3. Results
Across a held-out set of research papers, grounded answers with page citations were rated
as more trustworthy than ungrounded answers by human reviewers,

## Next

Phase 2 embeds these chunks and stores them in Qdrant, so this exact
`metadata["page_label"]` field becomes the page number shown in citations.